# Chapter 12: Explainable Agent

**Book:** *30 Agents Every AI Engineer Must Build*
**Author:** Imran Ahmad — Packt Publishing
**Chapter:** 12 — Ethical and Explainable Agents (p.23–39)

---

This notebook implements the **Explainable Agent** architecture and the
**Medical Diagnosis Assistant with Explanation** case study. Topics covered:

- Reasoning transparency with immutable audit trails (p.24–25)
- SHAP and LIME feature attribution frameworks (p.26)
- Counterfactual analysis for minimal-change explanations (p.27)
- Confidence communication with calibration and qualifier mapping (p.28–29)
- DiagnosticAssistant: biometrics → symptoms → differentials → explanation (p.30–35)
- Audience-adapted explanations — clinician vs. patient (p.34–35)
- Production failure mode demonstrations (p.35)
- Governance reference: Tables 12.2 and 12.3 (p.36–37)

> **Simulation Mode:** This notebook runs fully without an API key. All LLM calls
> are handled by `MockLLM` with chapter-derived, deterministic responses.


In [ ]:
# ============================================================================
# Cell 2: Setup — Imports, sys.path, and Mode Detection
# Chapter 12, p.2 (Tech Requirements)
# Author: Imran Ahmad
# ============================================================================

import sys
import os
from pathlib import Path

# Ensure the project root is on sys.path
project_root = str(Path.cwd().parent) if Path.cwd().name == "notebooks" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Matplotlib inline for notebook plots
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Core imports from src/
from src.utils import ColorLogger, graceful_fallback, resolve_api_key, get_mode, is_simulation
from src.mock_llm import MockLLM, get_mock_llm
from src.synthetic_data import generate_medical_dataset
from src.explainability_core import (
    DecisionLogger,
    ExplainableAgent,
    compute_shap_explanations,
    compute_lime_explanations,
    train_diagnostic_model,
    generate_counterfactual,
    TemperatureScaler,
    ConfidenceAwareAgent,
    BiometricAnalyzer,
    SymptomInterpreter,
    DiagnosticCoordinator,
    ClinicalExplainer,
    DiagnosticAssistant,
)

# Initialize logger and resolve mode
logger = ColorLogger(name="Notebook02")
api_key = resolve_api_key(interactive=True)
mode = get_mode()

logger.info(f"Notebook 02 ready. Mode: {mode}")
logger.info(f"NumPy: {np.__version__}, Pandas: {pd.__version__}")


## Reasoning Transparency (p.24–25)

The `ExplainableAgent` implements a **four-step decision process**, each step
logged to an immutable `DecisionLogger`:

1. **Input Reception** — Validate and record incoming data.
2. **Reasoning** — Analyze input, produce intermediate results.
3. **Decision** — Generate the final output with a confidence score.
4. **Explanation** — Produce a human-readable narrative of the decision.

The audit trail (`get_trace()`) returns a deep copy — it cannot be modified
after the fact. This is critical for regulatory compliance and post-hoc review.


In [ ]:
# ============================================================================
# Cell 4: ExplainableAgent — Four-Step Decision Pipeline
# Chapter 12, p.24–25 (Reasoning Transparency)
# Author: Imran Ahmad
# ============================================================================

agent = ExplainableAgent(name="MedicalReasoningAgent")

# Simulate a patient input
patient_data = {
    "heart_rate_avg": 88.5,
    "spo2_min": 93.2,
    "wbc_count": 12.4,
    "temperature": 38.1,
    "reported_symptoms": "productive cough, fever, shortness of breath",
}

# Run the full four-step pipeline
result = agent.run_full_pipeline(patient_data)

logger.success(f"Decision: {result['decision']['recommendation']}")
logger.success(f"Confidence: {result['decision']['confidence']:.0%}")
logger.info(f"Explanation: {result['explanation']}")


In [ ]:
# ============================================================================
# Cell 5: Audit Trail — Immutable Trace Recording
# Chapter 12, p.24–25 (DecisionLogger)
# Author: Imran Ahmad
# ============================================================================

trace = agent.get_trace()

logger.info(f"Trace summary: {agent.get_trace_summary()}")
logger.info(f"\n=== Full Audit Trail ({len(trace)} entries) ===")
for entry in trace:
    logger.info(
        f"  [{entry['timestamp'][:19]}] {entry['step']:25s} → {entry['status']}"
    )

# Demonstrate immutability — modifying the returned trace does NOT affect the original
trace[0]["step"] = "TAMPERED"
original_trace = agent.get_trace()
logger.success(
    f"\nImmutability check: original first step is still "
    f"'{original_trace[0]['step']}' (not 'TAMPERED')."
)


## LIME & SHAP Frameworks (p.26)

Two complementary approaches to feature attribution:

| Framework | Scope | Method |
|---|---|---|
| **SHAP** (SHapley Additive exPlanations) | Global + Local | Game-theoretic feature importance |
| **LIME** (Local Interpretable Model-agnostic Explanations) | Local only | Perturbed samples + linear surrogate |

We train a `GradientBoostingClassifier` on the synthetic medical dataset and
apply both SHAP and LIME to explain its predictions. All computation is wrapped
in `@graceful_fallback` for resilience against import errors and timeouts.


In [ ]:
# ============================================================================
# Cell 7: Train Diagnostic Model + SHAP Explanations
# Chapter 12, p.26 (SHAP Framework)
# Author: Imran Ahmad
# ============================================================================

# Generate synthetic medical dataset
med_df = generate_medical_dataset(n=50, seed=42)

# Train the diagnostic model
model, X, y, feature_names, label_encoder = train_diagnostic_model(med_df)
logger.info(f"Classes: {list(label_encoder.classes_)}")

# Compute SHAP explanations
shap_result = compute_shap_explanations(model, X, feature_names=feature_names)

if shap_result["status"] == "complete":
    logger.success("SHAP computation complete.")
    logger.info("\n=== Global Feature Importance (SHAP) ===")
    for feat, imp in shap_result["feature_importance"].items():
        bar = "█" * int(imp * 50)
        logger.info(f"  {feat:25s} {imp:.4f}  {bar}")

    # SHAP summary bar plot
    fig, ax = plt.subplots(figsize=(8, 4))
    feats = list(shap_result["feature_importance"].keys())
    vals = list(shap_result["feature_importance"].values())
    colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(feats)))
    ax.barh(feats[::-1], vals[::-1], color=colors[::-1])
    ax.set_xlabel("Mean |SHAP Value|")
    ax.set_title("SHAP Global Feature Importance (p.26)")
    plt.tight_layout()
    plt.show()
else:
    logger.error(f"SHAP unavailable: {shap_result['status']}")


In [ ]:
# ============================================================================
# Cell 8: LIME Explanations — Local Instance
# Chapter 12, p.26 (LIME Framework)
# Author: Imran Ahmad
# ============================================================================

# Explain instance 0 with LIME
lime_result = compute_lime_explanations(
    model, X, instance_idx=0,
    feature_names=feature_names, num_features=5
)

if lime_result["status"] == "complete":
    logger.success(f"LIME explanation for instance {lime_result['instance_idx']}:")
    logger.info("\n=== LIME Feature Weights ===")
    for feat, weight in lime_result["feature_weights"].items():
        direction = "↑ supports" if weight > 0 else "↓ opposes"
        logger.info(f"  {feat:40s} {weight:+.4f}  ({direction})")

    # LIME bar chart
    fig, ax = plt.subplots(figsize=(8, 4))
    feats_l = list(lime_result["feature_weights"].keys())
    weights_l = list(lime_result["feature_weights"].values())
    colors_l = ["#51CF66" if w > 0 else "#FF6B6B" for w in weights_l]
    ax.barh(feats_l[::-1], weights_l[::-1], color=colors_l[::-1])
    ax.axvline(x=0, color="gray", linewidth=0.8)
    ax.set_xlabel("Feature Weight")
    ax.set_title(f"LIME Explanation — Instance {lime_result['instance_idx']} (p.26)")
    plt.tight_layout()
    plt.show()
else:
    logger.error(f"LIME unavailable: {lime_result['status']}")


## Counterfactual Analysis (p.27)

**Question:** *"What is the smallest change to this patient's data that would
change the diagnosis?"*

Counterfactual explanations are valuable because they provide **actionable insights**
— they tell the user exactly which features need to change and by how much.

The algorithm uses a greedy search: for each feature, it perturbs in the direction
that moves the prediction toward the target class, seeking the minimal change set.


In [ ]:
# ============================================================================
# Cell 10: Counterfactual Analysis — Worked Example
# Chapter 12, p.27 (Counterfactual Analysis)
# Author: Imran Ahmad
# ============================================================================

# Pick instance 0 and find what would change the diagnosis
instance = X.iloc[0].values
original_pred = model.predict(instance.reshape(1, -1))[0]
original_class = label_encoder.classes_[original_pred]
logger.info(f"Original prediction: {original_class} (class index {original_pred})")

# Find counterfactual to a different class
target_class = 1 if original_pred != 1 else 0
target_name = label_encoder.classes_[target_class]
logger.info(f"Target counterfactual class: {target_name} (index {target_class})")

cf_result = generate_counterfactual(
    model=model,
    instance=instance,
    feature_names=feature_names,
    target_class=target_class,
    feature_ranges={
        "heart_rate_avg": (40, 150),
        "spo2_min": (70, 100),
        "wbc_count": (2, 30),
        "temperature": (35, 42),
        "chest_imaging_num": (0, 3),
    },
    max_iterations=200,
    step_size=0.15,
)

logger.info(f"\nCounterfactual status: {cf_result['status']}")
logger.info(f"Iterations: {cf_result['iterations']}")

if cf_result["changes"]:
    logger.info("\n=== Required Feature Changes ===")

    # Build comparison table
    rows = []
    for feat, change_info in cf_result["changes"].items():
        rows.append({
            "Feature": feat,
            "Original": change_info["original"],
            "Counterfactual": change_info["counterfactual"],
            "Change": change_info["change"],
        })

    cf_table = pd.DataFrame(rows)
    logger.success(f"\n{cf_table.to_string(index=False)}")

    # Visualize changes
    fig, ax = plt.subplots(figsize=(8, 3))
    feats_cf = [r["Feature"] for r in rows]
    changes_cf = [r["Change"] for r in rows]
    colors_cf = ["#FF6B6B" if c < 0 else "#4C72B0" for c in changes_cf]
    ax.barh(feats_cf, changes_cf, color=colors_cf)
    ax.axvline(x=0, color="gray", linewidth=0.8)
    ax.set_xlabel("Feature Change")
    ax.set_title(f"Counterfactual: {original_class} → {target_name} (p.27)")
    plt.tight_layout()
    plt.show()
else:
    logger.debug("No feature changes needed or counterfactual not found.")


## Confidence Communication (p.28–30)

A responsible agent must **quantify and communicate its uncertainty**. The
`ConfidenceAwareAgent` implements:

1. **Multi-hypothesis generation** — rank competing diagnoses.
2. **Temperature-scaled calibration** — adjust raw scores for reliability.
3. **Qualifier mapping** (p.29):
   - > 0.9 → *"High confidence"*
   - > 0.7 → *"Moderate confidence"*
   - ≤ 0.7 → *"Low confidence — human review recommended"*
4. **Uncertainty type** — epistemic (reducible) vs. aleatoric (inherent).


In [ ]:
# ============================================================================
# Cell 12: ConfidenceAwareAgent — Calibration & Qualifiers
# Chapter 12, p.28–29 (Confidence Communication)
# Author: Imran Ahmad
# ============================================================================

confidence_agent = ConfidenceAwareAgent(seed=42)

# Simulate differentials from a diagnostic pipeline
differentials = [
    {"diagnosis": "pneumonia", "raw_score": 0.87},
    {"diagnosis": "bronchitis", "raw_score": 0.09},
    {"diagnosis": "atelectasis", "raw_score": 0.04},
]

scored = confidence_agent.score_differentials(differentials)

logger.info("=== Scored Differentials ===")
for s in scored:
    logger.info(
        f"  {s['answer']:20s}  conf={s['confidence']:.4f}  "
        f"→ {s['qualifier']}"
    )

# Uncertainty communication
uncertainty = confidence_agent.communicate_uncertainty(scored)
logger.success(f"\nUncertainty summary: {uncertainty}")


In [ ]:
# ============================================================================
# Cell 13: Confidence Calibration — Visual Demo
# Chapter 12, p.28–29 (TemperatureScaler)
# Author: Imran Ahmad
# ============================================================================

scaler = TemperatureScaler(seed=42)

# Calibrate a range of raw scores to show the effect
raw_scores = np.linspace(0.05, 0.95, 20)
calibrated_scores = [scaler.calibrate(s) for s in raw_scores]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Raw vs. Calibrated
axes[0].plot(raw_scores, calibrated_scores, "o-", color="#4C72B0", label="Calibrated")
axes[0].plot([0, 1], [0, 1], "--", color="gray", alpha=0.5, label="Perfect (identity)")
axes[0].axhline(y=0.9, color="#51CF66", linestyle=":", alpha=0.7, label="High threshold")
axes[0].axhline(y=0.7, color="#FFD43B", linestyle=":", alpha=0.7, label="Moderate threshold")
axes[0].set_xlabel("Raw Score")
axes[0].set_ylabel("Calibrated Score")
axes[0].set_title("Temperature Scaling Calibration (p.28)")
axes[0].legend(fontsize=8)

# Plot 2: Qualifier distribution for scored differentials
qual_colors = {
    "High confidence": "#51CF66",
    "Moderate confidence": "#FFD43B",
    "Low confidence — human review recommended": "#FF6B6B",
}
for s in scored:
    color = qual_colors.get(s["qualifier"], "gray")
    axes[1].barh(s["answer"], s["confidence"], color=color, edgecolor="white")
    axes[1].text(s["confidence"] + 0.02, s["answer"],
                 f"{s['confidence']:.0%} ({s['qualifier'].split()[0]})",
                 va="center", fontsize=9)

axes[1].axvline(x=0.9, color="#51CF66", linestyle=":", linewidth=2)
axes[1].axvline(x=0.7, color="#FFD43B", linestyle=":", linewidth=2)
axes[1].set_xlabel("Calibrated Confidence")
axes[1].set_title("Differential Confidence Qualifiers (p.29)")
axes[1].set_xlim(0, 1.1)

plt.tight_layout()
plt.show()


## DiagnosticAssistant — Medical Case Study (p.30–35)

The `DiagnosticAssistant` orchestrates a five-stage pipeline using specialized
sub-agents:

| Stage | Sub-Agent | Output |
|---|---|---|
| 1 | `BiometricAnalyzer` | Vital sign flags (normal/abnormal) |
| 2 | `SymptomInterpreter` | SNOMED CT codes with confidence |
| 3 | `DiagnosticCoordinator` | Ranked differential diagnoses |
| 4 | `ConfidenceAwareAgent` | Calibrated scores + qualifiers |
| 5 | `ClinicalExplainer` | Audience-adapted narrative |

Every stage is wrapped in `@graceful_fallback`. Sensor dropout (F7), model
failure (F10), and explanation failure (F5) are all handled gracefully.


In [ ]:
# ============================================================================
# Cell 15: Generate & Explore the Synthetic Medical Dataset
# Chapter 12, p.30–32 (Medical Diagnosis Context)
# Author: Imran Ahmad
# ============================================================================

med_df = generate_medical_dataset(n=50, seed=42)

logger.info(f"Dataset shape: {med_df.shape}")
logger.info(f"Columns: {list(med_df.columns)}")
logger.info(f"\nDiagnosis distribution:")
for diag, count in med_df["true_diagnosis"].value_counts().items():
    logger.info(f"  {diag}: {count}")

# Show a sample patient
sample_patient = med_df.iloc[5].to_dict()
logger.info("\n=== Sample Patient Record (P-0005) ===")
for k, v in sample_patient.items():
    logger.info(f"  {k}: {v}")


In [ ]:
# ============================================================================
# Cell 16: DiagnosticAssistant — Clinician Report
# Chapter 12, p.32–34 (Diagnostic Pipeline)
# Author: Imran Ahmad
# ============================================================================

assistant = DiagnosticAssistant(seed=42)

# Pick a patient with elevated WBC and consolidation for interesting output
interesting_patients = med_df[
    (med_df["wbc_count"] > 10) &
    (med_df["chest_imaging"] == "right_lower_consolidation")
]

if len(interesting_patients) > 0:
    patient = interesting_patients.iloc[0].to_dict()
else:
    patient = med_df.iloc[0].to_dict()

logger.info(f"Running diagnosis for patient {patient.get('patient_id', '?')}...")
logger.info(f"  WBC: {patient.get('wbc_count')}, Imaging: {patient.get('chest_imaging')}")
logger.info(f"  Symptoms: {patient.get('reported_symptoms')}")

# Run with clinician audience
report_clinician = assistant.run_diagnosis(patient, audience="clinician")

logger.success(f"\n=== Clinician Report ===")
logger.info(f"Status: {report_clinician['status']}")

# Biometrics
logger.info(f"\n--- Biometrics ---")
logger.info(f"  {report_clinician['biometrics']['summary']}")
for flag in report_clinician['biometrics']['flags']:
    logger.debug(f"  Flag: {flag['vital']} = {flag['value']} ({flag['status']}, "
                 f"normal: {flag['normal_range']})")

# Symptoms
logger.info(f"\n--- Interpreted Symptoms ---")
for sym in report_clinician['symptoms']:
    logger.info(f"  {sym['symptom']:25s} → {sym['snomed_code']}  "
                f"(conf: {sym['confidence']:.2f})")

# Differentials
logger.info(f"\n--- Differentials ---")
for diff in report_clinician['scored_differentials']:
    logger.info(f"  {diff['answer']:20s}  conf={diff['confidence']:.4f}  "
                f"→ {diff['qualifier']}")

# Narrative
logger.success(f"\n--- Clinician Narrative ---")
narrative = report_clinician['explanation']
if isinstance(narrative, dict):
    logger.info(narrative.get('narrative', 'N/A'))
else:
    logger.info(str(narrative))


In [ ]:
# ============================================================================
# Cell 17: DiagnosticAssistant — Patient Report
# Chapter 12, p.34–35 (Audience Adaptation)
# Author: Imran Ahmad
# ============================================================================

# Run the same patient with patient audience
report_patient = assistant.run_diagnosis(patient, audience="patient")

logger.success("=== Patient-Friendly Report ===")
narrative = report_patient['explanation']
if isinstance(narrative, dict):
    logger.info(narrative.get('narrative', 'N/A'))
else:
    logger.info(str(narrative))

# Uncertainty summary
logger.info(f"\n--- Uncertainty Summary ---")
logger.info(report_patient.get('uncertainty_summary', 'N/A'))


In [ ]:
# ============================================================================
# Cell 18: DiagnosticAssistant — Pipeline Trace
# Chapter 12, p.30–35 (Full Pipeline)
# Author: Imran Ahmad
# ============================================================================

trace = report_clinician.get('trace', [])
logger.info(f"=== Pipeline Trace ({len(trace)} steps) ===")
for step in trace:
    logger.info(f"  [{step['timestamp'][:19]}] {step['step']:30s} → {step['status']}")

# Visualize as a horizontal timeline
if trace:
    fig, ax = plt.subplots(figsize=(10, 2.5))
    step_names = [s["step"].replace("_", "\n") for s in trace]
    statuses = [s["status"] for s in trace]
    colors_t = ["#51CF66" if s == "complete" else "#FF6B6B" for s in statuses]

    for i, (name, color) in enumerate(zip(step_names, colors_t)):
        ax.barh(0, 1, left=i, color=color, edgecolor="white", height=0.5)
        ax.text(i + 0.5, 0, name, ha="center", va="center", fontsize=8,
                fontweight="bold")

    # Arrows between steps
    for i in range(len(trace) - 1):
        ax.annotate("", xy=(i + 1, 0.35), xytext=(i + 0.95, 0.35),
                     arrowprops=dict(arrowstyle="->", color="gray", lw=1.5))

    ax.set_xlim(-0.2, len(trace) + 0.2)
    ax.set_ylim(-0.5, 0.8)
    ax.set_title("DiagnosticAssistant Pipeline Trace (p.30-35)", fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


## Audience Adaptation (p.34–35)

The same diagnostic result can be presented differently depending on
the audience. Compare the **clinician** and **patient** explanations
side by side below.

- **Clinician:** Technical terminology, SHAP feature contributions, specific
  confidence scores, recommended follow-up tests.
- **Patient:** Plain language, key factors explained simply, reassurance,
  guidance on next steps.


In [ ]:
# ============================================================================
# Cell 19: Audience Adaptation — Side-by-Side Comparison
# Chapter 12, p.34–35 (Audience-Adapted Explanations)
# Author: Imran Ahmad
# ============================================================================

clinician_text = report_clinician['explanation']
patient_text = report_patient['explanation']

if isinstance(clinician_text, dict):
    clinician_narrative = clinician_text.get('narrative', 'N/A')
else:
    clinician_narrative = str(clinician_text)

if isinstance(patient_text, dict):
    patient_narrative = patient_text.get('narrative', 'N/A')
else:
    patient_narrative = str(patient_text)

# Display side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clinician panel
axes[0].set_facecolor("#F0F7FF")
axes[0].text(0.05, 0.95, "CLINICIAN REPORT", transform=axes[0].transAxes,
             fontsize=13, fontweight="bold", va="top", color="#2B5797")
axes[0].text(0.05, 0.80, clinician_narrative, transform=axes[0].transAxes,
             fontsize=9, va="top", wrap=True,
             bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.8))
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].axis("off")
axes[0].set_title("Clinician Audience (p.34)", fontweight="bold")

# Patient panel
axes[1].set_facecolor("#F0FFF0")
axes[1].text(0.05, 0.95, "PATIENT REPORT", transform=axes[1].transAxes,
             fontsize=13, fontweight="bold", va="top", color="#2B7A0B")
axes[1].text(0.05, 0.80, patient_narrative, transform=axes[1].transAxes,
             fontsize=9, va="top", wrap=True,
             bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.8))
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].axis("off")
axes[1].set_title("Patient Audience (p.35)", fontweight="bold")

plt.suptitle("Audience-Adapted Explanations (p.34-35)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================================
# Cell 20: Production Failure Mode Demonstrations
# Chapter 12, p.35 (Production Failures)
# Author: Imran Ahmad
# ============================================================================

logger.info("=== Failure Mode Demonstrations ===")

# F7: Sensor Dropout — Patient record missing biometrics
logger.info("\n--- F7: Sensor Dropout (p.35) ---")
incomplete_patient = {
    "patient_id": "P-MISSING",
    "reported_symptoms": "fever, chest pain",
    "chest_imaging": "right_lower_consolidation",
    # Missing: heart_rate_avg, spo2_min, wbc_count, temperature
}
biometric_analyzer = BiometricAnalyzer()
result_f7 = biometric_analyzer.analyze(incomplete_patient)
logger.info(f"  Biometric flags: {len(result_f7['flags'])} (expected: 0, no data to flag)")
logger.info(f"  Summary: {result_f7['summary']}")

# F10: Model Failure — Force an error in explanation
logger.info("\n--- F10: Model Serving Failure Simulation (p.35) ---")

@graceful_fallback(
    fallback_value={"narrative": "Rule-based triage: present to emergency department.",
                    "source": "fallback"},
    section_ref="Section 12 - Model Failure (p.35)"
)
def failing_explanation():
    raise ConnectionError("Diagnostic model endpoint unreachable (simulated)")

result_f10 = failing_explanation()
logger.info(f"  Fallback narrative: {result_f10['narrative']}")

# F5: SHAP Timeout Simulation
logger.info("\n--- F5: SHAP Timeout Simulation (p.35) ---")

@graceful_fallback(
    fallback_value={"feature_importance": {"wbc_count": 0.31, "chest_imaging": 0.28},
                    "status": "timeout_fallback"},
    section_ref="Section 12 - SHAP Timeout (p.35)"
)
def slow_shap():
    raise TimeoutError("SHAP computation exceeded 60 second limit (simulated)")

result_f5 = slow_shap()
logger.info(f"  Fallback features: {result_f5['feature_importance']}")
logger.info(f"  Status: {result_f5['status']}")

logger.success("\nAll failure modes handled gracefully — notebook continues running.")


## Governance Reference (p.36–37)

### Table 12.2 — Explainability Requirements by Stakeholder

| Stakeholder | Required Information | Format | Frequency |
|---|---|---|---|
| Regulators | Full audit trail, fairness metrics, model documentation | Structured report | On request |
| Clinical staff | Differential diagnosis, confidence levels, key features | Clinician narrative | Per patient |
| Patients | Plain-language explanation, uncertainty, next steps | Patient narrative | Per visit |
| Engineers | Model performance, SHAP/LIME values, error rates | Dashboard + logs | Continuous |
| Ethics board | Bias reports, fairness trends, incident history | Quarterly review | Quarterly |

### Table 12.3 — Governance Checklist for Deployed Agents

| Requirement | Implementation in This Chapter |
|---|---|
| Audit trail | `DecisionLogger` with immutable trace (p.24-25) |
| Bias monitoring | `BiasMonitoringPipeline` with alerts (p.17-19) |
| Explanation | `ClinicalExplainer` with audience adaptation (p.34-35) |
| Human oversight | `escalate_to_human()` for critical decisions (p.9) |
| Fairness enforcement | `FairnessEnforcer` with three strategies (p.22-23) |
| Confidence communication | `ConfidenceAwareAgent` with qualifiers (p.28-29) |
| Failure handling | `@graceful_fallback` on all external calls (p.35) |
| Regulatory compliance | `EUCompliantAgent` with 7 requirements (p.10-11) |


## Summary & Exercises

### Key Takeaways

1. **Reasoning transparency** (p.24–25) requires an immutable audit trail. Every step
   in the decision process must be logged, timestamped, and retrievable.

2. **SHAP and LIME** (p.26) provide complementary views: SHAP gives global and local
   importance through game theory; LIME gives local explanations via surrogate models.

3. **Counterfactual analysis** (p.27) answers "what would need to change?" — the most
   actionable form of explanation for both clinicians and patients.

4. **Confidence communication** (p.28–29) must distinguish epistemic uncertainty
   (reducible with more data) from aleatoric uncertainty (inherent noise). The qualifier
   mapping ensures consistent language.

5. **Audience adaptation** (p.34–35) is essential: a clinician needs SHAP values and
   SNOMED codes; a patient needs plain language and reassurance.

6. **Production resilience** (p.35) demands `@graceful_fallback` on every external call.
   The system must never crash — it degrades gracefully and logs every failure.

### Suggested Exercises

- **Exercise 1:** Train the diagnostic model with a different algorithm (e.g.,
  `RandomForestClassifier`) and compare SHAP feature importance rankings.
- **Exercise 2:** Generate counterfactuals for 5 different patients and identify
  which features are most commonly changed.
- **Exercise 3:** Add a third audience level ("administrator") to `ClinicalExplainer`
  that includes cost and resource utilization information.
- **Exercise 4:** Implement a rolling SHAP monitor that tracks feature importance
  drift over batches of patients.

---

*Author: Imran Ahmad — Packt Publishing*
*Chapter 12: Ethical and Explainable Agents (p.23–39)*
